# 12 Clinical Variant

This notebook runs the full-sequence ClinVar variant workflow for the 14-3-3 predictor.

Current goal:
- start from ClinVar missense variants
- map each variant to a reviewed human UniProt protein sequence
- build the mutant full-length protein sequence
- run `predict_all_sites()` on both wild-type and mutant full-length sequences
- compare all S/T site predictions between WT and mutant

This notebook does not restrict analysis to mutation +/- 7 aa windows. The full sequence is rescored.

In [24]:
from pathlib import Path
import importlib

import pandas as pd

import clinical_variant_pipeline as cvp

cvp = importlib.reload(cvp)

CLINVAR_PROTEIN_PATTERN = cvp.CLINVAR_PROTEIN_PATTERN
load_clinvar_variants = cvp.load_clinvar_variants
load_geneid_to_uniprot_map = cvp.load_geneid_to_uniprot_map
load_uniprot_sequences = cvp.load_uniprot_sequences
run_variant_prediction_pipeline = cvp.run_variant_prediction_pipeline


In [25]:
variant_summary_path = Path("data/variant_summary.txt.gz")
idmapping_path = Path("data/idmapping_2024_05_02.xlsx")
fasta_path = Path("data/uniprotkb_reviewed_true_AND_model_organ_2024_12_23.fasta")
phospholingo_model_path = Path("/Users/newuser/PhosphoLingo_ST_new.ckpt")
esm_max_sequence_length = 1022

significance_filter = "pathogenic_or_likely_pathogenic"
require_germline = True
require_grch = True
deduplicate_by = "protein_change"
uniprot_resolution_policy = "first_matching"

run_mode = "smoke_test"  # change to "full_run" for the full dataset

if run_mode == "smoke_test":
    output_dir = Path("clinvar_smoke_test_output_v2")
    selected_variation_ids = []
    selected_gene_ids = []
    resume_existing_run = False
    max_variants = 2
elif run_mode == "full_run":
    output_dir = Path("clinvar_full_sequence_output_v2")
    selected_variation_ids = []
    selected_gene_ids = []
    resume_existing_run = False
    max_variants = None
else:
    raise ValueError("run_mode must be 'smoke_test' or 'full_run'")

# Optional targeted overrides:
# selected_variation_ids = ["925574", "925575"]
# selected_gene_ids = ["7157"]

variant_summary_path, idmapping_path, fasta_path, phospholingo_model_path, output_dir, esm_max_sequence_length, significance_filter, require_germline, require_grch, deduplicate_by, selected_variation_ids, selected_gene_ids, uniprot_resolution_policy, run_mode, resume_existing_run, max_variants


(PosixPath('data/variant_summary.txt.gz'),
 PosixPath('data/idmapping_2024_05_02.xlsx'),
 PosixPath('data/uniprotkb_reviewed_true_AND_model_organ_2024_12_23.fasta'),
 PosixPath('/Users/newuser/PhosphoLingo_ST_new.ckpt'),
 PosixPath('clinvar_smoke_test_output_v2'),
 1022,
 'pathogenic_or_likely_pathogenic',
 True,
 True,
 'protein_change',
 [],
 [],
 'first_matching',
 'smoke_test',
 False,
 2)

## Explore Raw ClinVar Variant Summary

Use this section first if you want to inspect the raw `variant_summary.txt.gz` table and adjust filtering logic before running the full pipeline.


In [26]:
raw_clinvar_df = pd.read_csv(variant_summary_path, sep="\t", low_memory=False)
print("Raw ClinVar rows:", len(raw_clinvar_df))
print("Columns:")
list(raw_clinvar_df.columns)


Raw ClinVar rows: 8982609
Columns:


['#AlleleID',
 'Type',
 'Name',
 'GeneID',
 'GeneSymbol',
 'HGNC_ID',
 'ClinicalSignificance',
 'ClinSigSimple',
 'LastEvaluated',
 'RS# (dbSNP)',
 'nsv/esv (dbVar)',
 'RCVaccession',
 'PhenotypeIDS',
 'PhenotypeList',
 'Origin',
 'OriginSimple',
 'Assembly',
 'ChromosomeAccession',
 'Chromosome',
 'Start',
 'Stop',
 'ReferenceAllele',
 'AlternateAllele',
 'Cytogenetic',
 'ReviewStatus',
 'NumberSubmitters',
 'Guidelines',
 'TestedInGTR',
 'OtherIDs',
 'SubmitterCategories',
 'VariationID',
 'PositionVCF',
 'ReferenceAlleleVCF',
 'AlternateAlleleVCF',
 'SomaticClinicalImpact',
 'SomaticClinicalImpactLastEvaluated',
 'ReviewStatusClinicalImpact',
 'Oncogenicity',
 'OncogenicityLastEvaluated',
 'ReviewStatusOncogenicity',
 'SCVsForAggregateGermlineClassification',
 'SCVsForAggregateSomaticClinicalImpact',
 'SCVsForAggregateOncogenicityClassification']

In [27]:
raw_clinvar_df.head()


,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
0,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTAACTGTAAA (p.Arg27_Ile28delinsLeuLeuTer),9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
1,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTAACTGTAAA (p.Arg27_Ile28delinsLeuLeuTer),9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
2,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
3,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
4,15043,single nucleotide variant,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),9640,ZNF592,HGNC:28986,Uncertain significance,0,"Jun 29, 2015",150829393,...,A,-,-,-,-,-,-,SCV000020157,-,-


In [28]:
raw_clinvar_df.shape

(8982609, 43)

In [29]:
raw_clinvar_df["Type"].value_counts(dropna=False).head(20) if "Type" in raw_clinvar_df.columns else pd.Series(dtype="int64")


Type
single nucleotide variant    8274459
Deletion                     347145 
Duplication                  148924 
Microsatellite               77556  
Indel                        38219  
copy number gain             32644  
copy number loss             30397  
Insertion                    28452  
Inversion                    3134   
Variation                    1128   
Translocation                352    
Complex                      100    
protein only                 93     
fusion                       5      
Tandem duplication           1      
Name: count, dtype: int64

In [30]:
raw_clinvar_df["ClinicalSignificance"].value_counts(dropna=False).head(30) if "ClinicalSignificance" in raw_clinvar_df.columns else pd.Series(dtype="int64")


ClinicalSignificance
Uncertain significance                                 4673771
Likely benign                                          2182660
-                                                      490980 
Benign                                                 425638 
Pathogenic                                             404641 
Conflicting classifications of pathogenicity           327537 
Likely pathogenic                                      241047 
Benign/Likely benign                                   129133 
Pathogenic/Likely pathogenic                           79760  
not provided                                           14806  
drug response                                          3802   
other                                                  3067   
no classification for the single variant               1407   
risk factor                                            769    
association                                            692    
conflicting data from submitters  

In [31]:
raw_clinvar_df["Assembly"].value_counts(dropna=False) if "Assembly" in raw_clinvar_df.columns else pd.Series(dtype="int64")


Assembly
GRCh37    4510851
GRCh38    4457595
na        9392   
NCBI36    4771   
Name: count, dtype: int64

In [32]:
protein_name_mask = raw_clinvar_df["Name"].astype(str).str.contains(CLINVAR_PROTEIN_PATTERN, regex=True)
print("Rows with protein-style variant names:", int(protein_name_mask.sum()))
raw_clinvar_df.loc[protein_name_mask, ["Name", "Type", "ClinicalSignificance"]].head(20)


/var/folders/jl/gnkp_hvs3zqb957750yclnqr0000gn/T/ipykernel_15105/768907162.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  protein_name_mask = raw_clinvar_df["Name"].astype(str).str.contains(CLINVAR_PROTEIN_PATTERN, regex=True)


Rows with protein-style variant names: 5217161


,Name,Type,ClinicalSignificance
4,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),single nucleotide variant,Uncertain significance
5,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),single nucleotide variant,Uncertain significance
6,NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter),single nucleotide variant,Pathogenic
7,NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter),single nucleotide variant,Pathogenic
8,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),single nucleotide variant,Likely pathogenic
9,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),single nucleotide variant,Likely pathogenic
10,NM_025152.3(NUBPL):c.166G>A (p.Gly56Arg),single nucleotide variant,Conflicting classifications of pathogenicity
11,NM_025152.3(NUBPL):c.166G>A (p.Gly56Arg),single nucleotide variant,Conflicting classifications of pathogenicity
12,NM_000410.4(HFE):c.845G>A (p.Cys282Tyr),single nucleotide variant,Conflicting classifications of pathogenicity; other; risk factor
13,NM_000410.4(HFE):c.845G>A (p.Cys282Tyr),single nucleotide variant,Conflicting classifications of pathogenicity; other; risk factor


In [33]:
custom_preview_df = raw_clinvar_df.copy()
custom_preview_df = custom_preview_df[custom_preview_df["Type"] == "single nucleotide variant"]
custom_preview_df = custom_preview_df[custom_preview_df["Name"].astype(str).str.contains(CLINVAR_PROTEIN_PATTERN, regex=True)]
if require_germline and "OriginSimple" in custom_preview_df.columns:
    custom_preview_df = custom_preview_df[custom_preview_df["OriginSimple"].astype(str).str.contains("germline", case=False, na=False)]
if require_grch and "Assembly" in custom_preview_df.columns:
    custom_preview_df = custom_preview_df[custom_preview_df["Assembly"].astype(str).str.contains("GRCh", na=False)]
custom_preview_df = cvp._apply_clinical_significance_filter(custom_preview_df, significance_filter)
filtered_before_dedup = len(custom_preview_df)
custom_preview_df = cvp._deduplicate_clinvar_variants(custom_preview_df, deduplicate_by)
print("Custom preview rows:", len(custom_preview_df))
print("Rows before dedup:", filtered_before_dedup)
print("Rows after dedup:", len(custom_preview_df))
custom_preview_df[["GeneID", "Name", "ClinicalSignificance", "OriginSimple", "Assembly"]].head(20)


/var/folders/jl/gnkp_hvs3zqb957750yclnqr0000gn/T/ipykernel_15105/4047400013.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  custom_preview_df = custom_preview_df[custom_preview_df["Name"].astype(str).str.contains(CLINVAR_PROTEIN_PATTERN, regex=True)]


Custom preview rows: 265475
Rows before dedup: 265475
Rows after dedup: 265475


,GeneID,Name,ClinicalSignificance,OriginSimple,Assembly
6,55572,NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter),Pathogenic,germline,GRCh37
7,55572,NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter),Pathogenic,germline,GRCh38
8,55572,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),Likely pathogenic,germline,GRCh37
9,55572,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),Likely pathogenic,germline,GRCh38
30,3077,NM_000410.4(HFE):c.989G>T (p.Arg330Met),Pathogenic,germline,GRCh37
31,3077,NM_000410.4(HFE):c.989G>T (p.Arg330Met),Pathogenic,germline,GRCh38
32,3077,NM_000410.4(HFE):c.848A>C (p.Gln283Pro),Pathogenic/Likely pathogenic,germline,GRCh37
33,3077,NM_000410.4(HFE):c.848A>C (p.Gln283Pro),Pathogenic/Likely pathogenic,germline,GRCh38
36,57539,NM_020779.4(WDR35):c.1844A>G (p.Glu615Gly),Pathogenic,germline,GRCh37
37,57539,NM_020779.4(WDR35):c.1844A>G (p.Glu615Gly),Pathogenic,germline,GRCh38


## Preview ClinVar Variants

This step filters to protein-level missense SNVs and extracts the amino-acid change.

In [34]:
clinvar_df = load_clinvar_variants(
    variant_summary_path=variant_summary_path,
    significance_filter=significance_filter,
    require_germline=require_germline,
    require_grch=require_grch,
    deduplicate_by=deduplicate_by,
    selected_variation_ids=selected_variation_ids or None,
    selected_gene_ids=selected_gene_ids or None,
)

print("ClinVar variants after filtering:", len(clinvar_df))
clinvar_df[[
    "GeneID",
    "Name",
    "ClinicalSignificance",
    "ref_aa",
    "position",
    "alt_aa",
]].head()


/Users/newuser/1433predictor/clinical_variant_pipeline.py:176: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  clinvar_df = clinvar_df[clinvar_df["Name"].astype(str).str.contains(CLINVAR_PROTEIN_PATTERN, regex=True)].copy()


ClinVar variants after filtering: 65793


,GeneID,Name,ClinicalSignificance,ref_aa,position,alt_aa
0,55572,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),Likely pathogenic,N,430,S
1,3077,NM_000410.4(HFE):c.989G>T (p.Arg330Met),Pathogenic,R,330,M
2,3077,NM_000410.4(HFE):c.848A>C (p.Gln283Pro),Pathogenic/Likely pathogenic,Q,283,P
3,57539,NM_020779.4(WDR35):c.1844A>G (p.Glu615Gly),Pathogenic,E,615,G
4,57539,NM_020779.4(WDR35):c.2590G>A (p.Ala864Thr),Likely pathogenic,A,864,T


In [35]:
clinvar_df

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,ref_aa3,position,alt_aa3,ref_aa,alt_aa
0,15045,single nucleotide variant,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),55572,FOXRED1,HGNC:26927,Likely pathogenic,1,"Jun 06, 2024",267606830,...,-,-,SCV005680614,-,-,Asn,430,Ser,N,S
1,15057,single nucleotide variant,NM_000410.4(HFE):c.989G>T (p.Arg330Met),3077,HFE,HGNC:4886,Pathogenic,1,"Aug 01, 1999",111033558,...,-,-,SCV000020178,-,-,Arg,330,Met,R,M
2,15058,single nucleotide variant,NM_000410.4(HFE):c.848A>C (p.Gln283Pro),3077,HFE,HGNC:4886,Pathogenic/Likely pathogenic,1,"Jun 11, 2025",111033563,...,-,-,SCV001214178|SCV004028688|SCV004702708|SCV005875314,-,-,Gln,283,Pro,Q,P
3,15060,single nucleotide variant,NM_020779.4(WDR35):c.1844A>G (p.Glu615Gly),57539,WDR35,HGNC:29250,Pathogenic,1,"Sep 10, 2010",267607174,...,-,-,SCV000020181,-,-,Glu,615,Gly,E,G
4,15062,single nucleotide variant,NM_020779.4(WDR35):c.2590G>A (p.Ala864Thr),57539,WDR35,HGNC:29250,Likely pathogenic,1,"May 25, 2017",267607175,...,-,-,SCV000605607,-,-,Ala,864,Thr,A,T
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65788,4963054,single nucleotide variant,NM_005670.4(EPM2A):c.299A>G (p.Glu100Gly),7957,EPM2A,HGNC:3413,Likely pathogenic,1,"Apr 21, 2026",-1,...,-,-,SCV007541392,-,-,Glu,100,Gly,E,G
65789,4963055,single nucleotide variant,NM_005670.4(EPM2A):c.830T>G (p.Val277Gly),7957,EPM2A,HGNC:3413,Likely pathogenic,1,"Apr 21, 2026",-1,...,-,-,SCV007541391,-,-,Val,277,Gly,V,G
65790,4963056,single nucleotide variant,NM_000338.3(SLC12A1):c.1685C>T (p.Ala562Val),6557,SLC12A1,HGNC:10910,Likely pathogenic,1,"Apr 21, 2026",-1,...,-,-,SCV007541389,-,-,Ala,562,Val,A,V
65791,4963058,single nucleotide variant,NM_001355436.2(SPTB):c.251A>T (p.Asp84Val),6710,SPTB,HGNC:11274,Likely pathogenic,1,"Apr 21, 2026",-1,...,-,-,SCV007541406,-,-,Asp,84,Val,D,V


In [36]:
clinvar_df["GeneID"].value_counts()

GeneID
2200      1426
6323      950 
3949      690 
1287      650 
24        632 
         ...  
79966     1   
122830    1   
23265     1   
51053     1   
4892      1   
Name: count, Length: 4303, dtype: int64

In [37]:
clinvar_df["ClinicalSignificance"].value_counts(dropna=False)


ClinicalSignificance
Likely pathogenic                               30989
Pathogenic                                      22347
Pathogenic/Likely pathogenic                    12258
Likely pathogenic, low penetrance               63   
Pathogenic; other                               56   
Pathogenic; drug response                       37   
Likely pathogenic; drug response                9    
Pathogenic; risk factor                         5    
Pathogenic, low penetrance                      5    
Likely pathogenic; association                  4    
Pathogenic/Likely pathogenic; other             4    
Pathogenic/Likely pathogenic; risk factor       4    
Pathogenic; Affects                             3    
Pathogenic; association                         3    
Pathogenic/Likely pathogenic, low penetrance    2    
Pathogenic/Likely pathogenic; drug response     1    
Pathogenic/Likely pathogenic; association       1    
Likely pathogenic; risk factor                  1    
Likely 

## Preview UniProt Mapping

This step loads the reviewed human UniProt map and the reviewed human FASTA sequences.

In [38]:
geneid_to_uniprot = load_geneid_to_uniprot_map(idmapping_path)
sequence_by_accession = load_uniprot_sequences(fasta_path)

print("GeneID to UniProt entries:", len(geneid_to_uniprot))
print("Reviewed UniProt sequences loaded:", len(sequence_by_accession))


/Users/newuser/anaconda3/envs/1433predictor2026/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


GeneID to UniProt entries: 18137
Reviewed UniProt sequences loaded: 20426


In [39]:
preview_df = clinvar_df.copy()
preview_df["uniprot_ids"] = preview_df["GeneID"].map(geneid_to_uniprot)
preview_df[[
    "GeneID",
    "Name",
    "ref_aa",
    "position",
    "alt_aa",
    "uniprot_ids",
]].head(10)


,GeneID,Name,ref_aa,position,alt_aa,uniprot_ids
0,55572,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),N,430,S,[Q96CU9]
1,3077,NM_000410.4(HFE):c.989G>T (p.Arg330Met),R,330,M,[Q30201]
2,3077,NM_000410.4(HFE):c.848A>C (p.Gln283Pro),Q,283,P,[Q30201]
3,57539,NM_020779.4(WDR35):c.1844A>G (p.Glu615Gly),E,615,G,[Q9P2L0]
4,57539,NM_020779.4(WDR35):c.2590G>A (p.Ala864Thr),A,864,T,[Q9P2L0]
5,112817,NM_138413.4(HOGA1):c.860G>T (p.Gly287Val),G,287,V,[Q86XE5]
6,112817,NM_138413.4(HOGA1):c.289C>T (p.Arg97Cys),R,97,C,[Q86XE5]
7,112817,NM_138413.4(HOGA1):c.209G>C (p.Arg70Pro),R,70,P,[Q86XE5]
8,112817,NM_138413.4(HOGA1):c.769T>G (p.Cys257Gly),C,257,G,[Q86XE5]
9,284403,NM_001083961.2(WDR62):c.671G>C (p.Trp224Ser),W,224,S,[O43379]


In [40]:
preview_df

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,ref_aa3,position,alt_aa3,ref_aa,alt_aa,uniprot_ids
0,15045,single nucleotide variant,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),55572,FOXRED1,HGNC:26927,Likely pathogenic,1,"Jun 06, 2024",267606830,...,-,SCV005680614,-,-,Asn,430,Ser,N,S,[Q96CU9]
1,15057,single nucleotide variant,NM_000410.4(HFE):c.989G>T (p.Arg330Met),3077,HFE,HGNC:4886,Pathogenic,1,"Aug 01, 1999",111033558,...,-,SCV000020178,-,-,Arg,330,Met,R,M,[Q30201]
2,15058,single nucleotide variant,NM_000410.4(HFE):c.848A>C (p.Gln283Pro),3077,HFE,HGNC:4886,Pathogenic/Likely pathogenic,1,"Jun 11, 2025",111033563,...,-,SCV001214178|SCV004028688|SCV004702708|SCV005875314,-,-,Gln,283,Pro,Q,P,[Q30201]
3,15060,single nucleotide variant,NM_020779.4(WDR35):c.1844A>G (p.Glu615Gly),57539,WDR35,HGNC:29250,Pathogenic,1,"Sep 10, 2010",267607174,...,-,SCV000020181,-,-,Glu,615,Gly,E,G,[Q9P2L0]
4,15062,single nucleotide variant,NM_020779.4(WDR35):c.2590G>A (p.Ala864Thr),57539,WDR35,HGNC:29250,Likely pathogenic,1,"May 25, 2017",267607175,...,-,SCV000605607,-,-,Ala,864,Thr,A,T,[Q9P2L0]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65788,4963054,single nucleotide variant,NM_005670.4(EPM2A):c.299A>G (p.Glu100Gly),7957,EPM2A,HGNC:3413,Likely pathogenic,1,"Apr 21, 2026",-1,...,-,SCV007541392,-,-,Glu,100,Gly,E,G,[O95278]
65789,4963055,single nucleotide variant,NM_005670.4(EPM2A):c.830T>G (p.Val277Gly),7957,EPM2A,HGNC:3413,Likely pathogenic,1,"Apr 21, 2026",-1,...,-,SCV007541391,-,-,Val,277,Gly,V,G,[O95278]
65790,4963056,single nucleotide variant,NM_000338.3(SLC12A1):c.1685C>T (p.Ala562Val),6557,SLC12A1,HGNC:10910,Likely pathogenic,1,"Apr 21, 2026",-1,...,-,SCV007541389,-,-,Ala,562,Val,A,V,[Q13621]
65791,4963058,single nucleotide variant,NM_001355436.2(SPTB):c.251A>T (p.Asp84Val),6710,SPTB,HGNC:11274,Likely pathogenic,1,"Apr 21, 2026",-1,...,-,SCV007541406,-,-,Asp,84,Val,D,V,[P11277]


## Stop Here Before the Heavy Step

The cells above only preview ClinVar variants and UniProt mapping. The full-sequence WT-vs-mutant prediction block below is the slowest part of the notebook, so keep it for last.


## Final Step: Run Full-Sequence WT vs Mutant Predictions

This is the slowest step in the notebook. Run the preview / mapping cells above first, and only run this section when you are ready to launch the full WT-vs-mutant sequence prediction workflow.

For each resolved ClinVar missense variant, this step:
- finds a matching WT reviewed UniProt sequence
- builds the full mutant sequence
- runs `predict_all_sites()` on the WT sequence
- runs `predict_all_sites()` on the mutant sequence
- saves both variant-level and site-level outputs


In [41]:
print(f"Running mode={run_mode} with output_dir={output_dir} resume={resume_existing_run} max_variants={max_variants}")
variant_summary_df, site_comparison_df, variant_output, site_output = run_variant_prediction_pipeline(
    variant_summary_path=variant_summary_path,
    idmapping_path=idmapping_path,
    fasta_path=fasta_path,
    output_dir=output_dir,
    significance_filter=significance_filter,
    require_germline=require_germline,
    require_grch=require_grch,
    deduplicate_by=deduplicate_by,
    selected_variation_ids=selected_variation_ids or None,
    selected_gene_ids=selected_gene_ids or None,
    uniprot_resolution_policy=uniprot_resolution_policy,
    phospholingo_model_loc=phospholingo_model_path,
    esm_max_sequence_length=esm_max_sequence_length,
    resume=resume_existing_run,
    max_variants=max_variants,
)

print("Variant summary saved to:", variant_output)
print("Site-level output saved to:", site_output)
print("Run stats saved to:", output_dir / "clinvar_run_stats.json")
print("Resolved variants:", (variant_summary_df["status"] == "ok").sum() if not variant_summary_df.empty else 0)
print("Variant rows:", len(variant_summary_df))
print("Site rows:", len(site_comparison_df))


Running mode=smoke_test with output_dir=clinvar_smoke_test_output_v2 resume=False max_variants=2


/Users/newuser/1433predictor/clinical_variant_pipeline.py:176: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  clinvar_df = clinvar_df[clinvar_df["Name"].astype(str).str.contains(CLINVAR_PROTEIN_PATTERN, regex=True)].copy()
/Users/newuser/anaconda3/envs/1433predictor2026/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/newuser/anaconda3/envs/1433predictor2026/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/newuser/1433predictor/src/files/DeePhase/__PREDICT/deephase_utils.py:448: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poo

--- Loaded /Users/newuser/1433predictor/temp_1549307efbc24b0c9e47280e5cac2103.fasta data ---
- Number of proteins: 16
- Number of positive sites: 0
- Number of negative sites: 16

--- Loaded /Users/newuser/1433predictor/temp_2694155a684342bcab121f72a17bef04.fasta data ---
- Number of proteins: 16
- Number of positive sites: 0
- Number of negative sites: 16

--- Loaded /Users/newuser/1433predictor/temp_a8c54c7d39dd481f8f2cc1c61e198ff9.fasta data ---
- Number of proteins: 5
- Number of positive sites: 0
- Number of negative sites: 5



/Users/newuser/anaconda3/envs/1433predictor2026/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/newuser/1433predictor/src/files/DeePhase/__PREDICT/deephase_utils.py:448: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_interm['phys_multi'] = predict_multiclass('phys_multi', data_phys_sel)['prediction_phys_multi']
/Users/newuser/anaconda3/envs/1433predictor2026/lib/python3.9/site-packages/sklearn/base.py:450: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/Users/newuser/1433predictor/src/files/DeePhase/__PREDICT/deephase_utils.py:449: PerformanceWarni

--- Loaded /Users/newuser/1433predictor/temp_891f8fccd84641a28a861023f8a46e87.fasta data ---
- Number of proteins: 16
- Number of positive sites: 0
- Number of negative sites: 16

--- Loaded /Users/newuser/1433predictor/temp_8b1b3114dd52400b98eddfce8c6675b9.fasta data ---
- Number of proteins: 16
- Number of positive sites: 0
- Number of negative sites: 16

--- Loaded /Users/newuser/1433predictor/temp_a1a9ebb8b4e44aef8e5b28c529509ebc.fasta data ---
- Number of proteins: 5
- Number of positive sites: 0
- Number of negative sites: 5

Variant summary saved to: clinvar_smoke_test_output_v2/clinvar_variant_summary_predictions.csv
Site-level output saved to: clinvar_smoke_test_output_v2/clinvar_site_level_predictions.csv
Run stats saved to: clinvar_smoke_test_output_v2/clinvar_run_stats.json
Resolved variants: 2
Variant rows: 2
Site rows: 90


## Inspect Variant-Level Results

In [42]:
variant_summary_df.head(20)


,GeneID,variation_id,name,clinical_significance,accession,mutation,mutation_position,sequence_length,esm_max_sequence_length,status,wt_site_count,wt_positive_site_count,wt_best_site,wt_best_probability,mut_site_count,mut_positive_site_count,mut_best_site,mut_best_probability
0,55572,6,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),Likely pathogenic,Q96CU9,N430S,430,NaN,NaN,ok,52,1,102,0.598451,53,1,102,0.598451
1,3077,18,NM_000410.4(HFE):c.989G>T (p.Arg330Met),Pathogenic,Q30201,R330M,330,NaN,NaN,ok,37,0,29,0.499462,37,0,29,0.448974


In [43]:
variant_summary_df[variant_summary_df["status"] == "ok"].sort_values(
    by=["mut_best_probability", "wt_best_probability"],
    ascending=[False, False],
).head(20)


,GeneID,variation_id,name,clinical_significance,accession,mutation,mutation_position,sequence_length,esm_max_sequence_length,status,wt_site_count,wt_positive_site_count,wt_best_site,wt_best_probability,mut_site_count,mut_positive_site_count,mut_best_site,mut_best_probability
0,55572,6,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),Likely pathogenic,Q96CU9,N430S,430,NaN,NaN,ok,52,1,102,0.598451,53,1,102,0.598451
1,3077,18,NM_000410.4(HFE):c.989G>T (p.Arg330Met),Pathogenic,Q30201,R330M,330,NaN,NaN,ok,37,0,29,0.499462,37,0,29,0.448974


## Inspect Site-Level WT vs Mutant Differences

`delta_positive_probability = mutant_probability - wild_type_probability`

In [44]:
site_comparison_df.head(20)


,site,wt_residue,wt_prediction,wt_predicted_positive,wt_positive_probability,mut_residue,mut_prediction,mut_predicted_positive,mut_positive_probability,accession,gene_id,mutation_position,mutation,delta_positive_probability,site_distance_to_mutation
0,16,T,0.0,False,0.044489,T,0,False,0.044489,Q96CU9,55572,430,N430S,0.000000,414
1,21,T,0.0,False,0.287988,T,0,False,0.287988,Q96CU9,55572,430,N430S,0.000000,409
2,27,S,0.0,False,0.155384,S,0,False,0.155384,Q96CU9,55572,430,N430S,0.000000,403
3,35,S,0.0,False,0.195874,S,0,False,0.195874,Q96CU9,55572,430,N430S,0.000000,395
4,43,S,0.0,False,0.071456,S,0,False,0.072489,Q96CU9,55572,430,N430S,0.001033,387
5,49,S,0.0,False,0.083438,S,0,False,0.079223,Q96CU9,55572,430,N430S,-0.004216,381
6,56,T,0.0,False,0.015930,T,0,False,0.015930,Q96CU9,55572,430,N430S,0.000000,374
7,57,S,0.0,False,0.050280,S,0,False,0.050280,Q96CU9,55572,430,N430S,0.000000,373
8,64,S,0.0,False,0.010980,S,0,False,0.010980,Q96CU9,55572,430,N430S,0.000000,366
9,77,S,0.0,False,0.013076,S,0,False,0.013057,Q96CU9,55572,430,N430S,-0.000019,353


In [45]:
site_comparison_df.assign(
    abs_delta=lambda df: df["delta_positive_probability"].abs()
).sort_values(
    by="abs_delta",
    ascending=False,
).head(50)


,site,wt_residue,wt_prediction,wt_predicted_positive,wt_positive_probability,mut_residue,mut_prediction,mut_predicted_positive,mut_positive_probability,accession,gene_id,mutation_position,mutation,delta_positive_probability,site_distance_to_mutation,abs_delta
89,335,S,0.0,False,0.473138,S,0,False,0.282253,Q30201,3077,330,R330M,-0.190885,5,0.190885
47,430,NaN,NaN,NaN,NaN,S,0,False,0.138453,Q96CU9,55572,430,N430S,0.138453,0,0.138453
88,311,S,0.0,False,0.075249,S,0,False,0.020874,Q30201,3077,330,R330M,-0.054376,19,0.054376
55,29,S,0.0,False,0.499462,S,0,False,0.448974,Q30201,3077,330,R330M,-0.050488,301,0.050488
81,236,T,0.0,False,0.088092,T,0,False,0.053105,Q30201,3077,330,R330M,-0.034987,94,0.034987
29,248,S,0.0,False,0.408968,S,0,False,0.431275,Q96CU9,55572,430,N430S,0.022307,182,0.022307
50,463,T,0.0,False,0.156533,T,0,False,0.175587,Q96CU9,55572,430,N430S,0.019054,33,0.019054
69,115,S,0.0,False,0.114287,S,0,False,0.095477,Q30201,3077,330,R330M,-0.018810,215,0.018810
65,90,S,0.0,False,0.131151,S,0,False,0.148270,Q30201,3077,330,R330M,0.017119,240,0.017119
30,249,S,0.0,False,0.071979,S,0,False,0.086461,Q96CU9,55572,430,N430S,0.014483,181,0.014483


In [46]:
if not site_comparison_df.empty:
    site_comparison_df[
        site_comparison_df["accession"] == site_comparison_df["accession"].iloc[0]
    ].sort_values("site").head(50)
else:
    pd.DataFrame()
